Gets the predicted values for scLEMBAS models.

Loss is assessed with EMD for single-cell measurements (for comparison to single-cell baselines random and no adversarial removal)

For psuedobulk (to compare to RF, linear, and train mean baselines), loss is calculated after psuedobulking either by perturbation or condition (cell type x perturbation) using the mean of the sample-wise RMSE (as described in [Ahlmann-Eltze et al](https://doi.org/10.1038/s41592-025-02772-6). Unlike McCauley, we don't implement the Pearson delta (used in [Ahlmann-Eltze et al](https://doi.org/10.1038/s41592-025-02772-6) and [Csendes et al](https://doi.org/10.1186/s12864-025-11600-2)) because both of the binary perturbations are present in test AND used in the counterfactuals (no good "control" vector). Instead, we use simple pearson.  

For prediction visualization, we project predictions into the PLS fit on the global actual data TF activity space from Notebook 04. 

In [1]:
import os
import collections
import itertools

import numpy as np
import pandas as pd

import copy

import sys
sys.path.insert(1, '../.')
from Kang_utils import all_data

sys.path.insert(1, '../../.') 
from notebook_utils import get_split, pb_y_pred

sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import io
from scLEMBAS.predict import (
    get_prediction, 
    merge_novar_predictions, 
    merge_ctrl_with_pert, 
    project_umap_global)

from scLEMBAS.metrics import distances 
from scLEMBAS import latent_separation as ls


/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install

In [2]:
seed = 888
data_path = '/home/hmbaghda/orcd/pool/scLEMBAS/analysis'
author = 'Kang'

n_cores = 30
os.environ["OMP_NUM_THREADS"] = str(1)
os.environ["MKL_NUM_THREADS"] = str(1)
os.environ["OPENBLAS_NUM_THREADS"] = str(1)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(1)
os.environ["NUMEXPR_NUM_THREADS"] = str(1)

In [3]:
(sn_ppis, tf_adata, adata, expr, source_label, target_label, weight_label, 
 stimulation_label, inhibition_label, cat_col, pert_col) = all_data

gt_map = {pert_col: 'perturbation', 'condition': 'condition'}

error_type = 'per_sample'

# Single-cell losses

Here, we get the fitted scLEMBAS model predictions. Next, we get the EMD for the scLEMBAS models (full model, random baseline, no adversarial training baseline). 

For the full model, it will also be compared to psuedobulk baselines (RF, linear, training mean), so we will also calculate the pseudo-bulk RMSE and Pearson delta metrics.

In [4]:
global_bias_removed = [['adj', 'global_bias'], 'global_bias', 'total_bias']

folds = range(5)
remove_types = [
    'none',
    ['adj', 'categorical_bias'],
    ['adj', 'global_bias'],
    'total_bias', 
    'adj',
    'categorical_bias',
    'global_bias'
]
mod_types = ['actual', 'random', 'noadv']

In [15]:
from Kang_utils import rev_stim
ctrl_pert = 'CTRL'

def prediction_pipeline(fold, mod_type, remove_type):
    split = get_split(fold = fold, author = author)

    test_conds = split['test_conds']
    train_barcodes = split['train_barcodes']
    test_barcodes = split['test_barcodes']

    # load trainer
    fn_trainer = os.path.join(data_path, 'processed', '{}_fold{}trainer_{}.pickle'.format(author, fold, mod_type))
    trainer = io.read_pickled_object(fn_trainer)
    mod = trainer.mod
    assert trainer.X_train.index.tolist() == train_barcodes, 'Train barcodes mismatch'
    assert trainer.X_test.index.tolist() == test_barcodes, 'Test barcodes mismatch'

    # get the control cells (from which counterfactual was predicted)
    ctrl_conds = sorted(set([tc.split('^')[0] + '^' + rev_stim[tc.split('^')[1]] for tc in test_conds]))
    ctrl_mask = tf_adata.obs.loc[train_barcodes, 'condition'].isin(ctrl_conds).values
    ctrl_cells = list(np.array(train_barcodes)[ctrl_mask])

    loss_res = collections.defaultdict(list)
    predictions_res = {}

    tf_adata_predicted = get_prediction(
        mod = mod,
        train_cells = train_barcodes,
        test_cells = test_barcodes, 
        tf_adata = tf_adata,
        cat_col = cat_col,
        pert_col = pert_col,
        ctrl_pert = ctrl_pert, 
        counterfactual = 'perturbation',
        cat_counterfactual_map = None,
        remove_type = remove_type,
        return_bias = False, 
        max_cells = int(5e3), 
        return_full = False, 
        stim_label_map = {'STIM': 'IFNB1', 'CTRL': 'CTRL'}, 
    )

    # calculate the losses
    emd_loss = distances.get_EMD_loss(tf_adata, tf_adata_predicted)['Mean EMD Loss']
    loss_res['loss'].append(emd_loss)
    loss_res['loss_type'].append('EMD')
    loss_res['remove_type'].append(remove_type)
    loss_res['fold'].append(fold)
    loss_res['model_type'].append(mod_type)
    loss_res['space'].append('feature')

    if mod_type == 'actual' and remove_type == 'none':
        iterable_pb_mets = itertools.product(
            [pert_col, 'condition'], 
            ['feature', 'pca']
        )
        for groupby_col, space in iterable_pb_mets:
            embedding_model = None if space == 'feature' else {'adata_pca': tf_adata}

            test_mask = tf_adata.obs.condition.isin(test_conds)
            rmse_loss = distances.get_rmse(tf_adata[test_mask], tf_adata_predicted, 
                                           groupby_col = groupby_col, error_type = error_type, 
                                           embedding_model = embedding_model
                                          )
            loss_res['loss'].append(rmse_loss)
            loss_res['loss_type'].append('RMSE_{}'.format(groupby_col))
            loss_res['remove_type'].append(remove_type)
            loss_res['fold'].append(fold)
            loss_res['model_type'].append(mod_type)
            loss_res['space'].append(space)

            pd_loss = distances.get_pearson(tf_adata[test_mask], tf_adata_predicted, 
                                            groupby_col = groupby_col, 
                                            error_type = error_type, embedding_model = embedding_model)
            loss_res['loss'].append(pd_loss)
            loss_res['loss_type'].append('Pearson_{}'.format(groupby_col))
            loss_res['remove_type'].append(remove_type)
            loss_res['fold'].append(fold)
            loss_res['model_type'].append(mod_type)
            loss_res['space'].append(space)

    # CTRL: get the control condition predictions (training data with no counterfactual) as a comparison
    tf_adata_ctrl_predicted = get_prediction(
        mod = mod,
        train_cells = ctrl_cells,
        test_cells = [], 
        tf_adata = tf_adata,
        cat_col = cat_col,
        pert_col = pert_col,
        ctrl_pert = ctrl_pert, 
        counterfactual = None,
        cat_counterfactual_map = None,
        remove_type = remove_type,
        return_bias = False, 
        max_cells = int(5e3), 
        return_full = False,
        stim_label_map = {'STIM': 'IFNB1', 'CTRL': 'CTRL'},
    )

    # check lack of variance in non-generative predictions (no b_g) and merge
    if remove_type in global_bias_removed:
        tf_adata_predicted = merge_novar_predictions(tf_adata_predicted, remove_type, 
                                                     cat_col = cat_col, pert_col = pert_col, 
                                                     atol = 1e-4, rtol = 1e-8)
        # CTRL:
        tf_adata_ctrl_predicted = merge_novar_predictions(tf_adata_ctrl_predicted, remove_type, 
                                                          cat_col = cat_col, pert_col = pert_col, 
                                                          atol = 1e-4, rtol = 1e-8)

    tf_adata_merged = merge_ctrl_with_pert(
        tf_adata_actual = tf_adata, 
        tf_adata_pert_predicted = tf_adata_predicted, 
        tf_adata_ctrl_predicted = tf_adata_ctrl_predicted, 
        cat_col = cat_col, 
        pert_col = pert_col
    )

    # use the global PCA space 
    X_pca = ls.project_to_pca(X_new = tf_adata_merged.X, adata = tf_adata)
    tf_adata_merged.obsm['X_pca'] = X_pca

    X_umap = project_umap_global(
        tf_adata_actual = tf_adata, 
        tf_adata_merged = tf_adata_merged, 
        only_project_predictions = False, # for consistency with how we project UMAP in McCauley
                                )
    tf_adata_merged.obsm['X_umap'] = X_pca

    remove_type_ = remove_type
    if type(remove_type) == list:
        remove_type_ = '^'.join(remove_type)    
    predictions_res[remove_type_ + '_{}'.format(fold)] = tf_adata_merged

    loss_res = pd.DataFrame(loss_res)
    loss_res.remove_type = loss_res.remove_type.apply(lambda x: '^'.join(x) if type(x) == list else x)

    return loss_res, predictions_res

In [16]:
iterables = list(itertools.product(mod_types, folds, remove_types))
# only do remove type not None for the full model
iterables = [it for it in iterables if ((it[0] == 'actual') or it[2] == 'none')]
n_tot_iter = len(iterables)

loss_res_all = []
predictions_res_all = []
predictions_res_noadv = []
predictions_res_rand = []
iter_counter = 0
for (mod_type, fold, remove_type) in iterables:
    loss_res, predictions_res = prediction_pipeline(fold = fold, mod_type = mod_type, remove_type = remove_type)
    loss_res_all.append(loss_res)

    if mod_type == 'actual':
        predictions_res_all.append(predictions_res)
    elif mod_type == 'noadv':
        predictions_res_noadv.append(predictions_res)
    elif mod_type == 'random':
        predictions_res_rand.append(predictions_res)

    print('------------------ {} of {} ------------------'.format(iter_counter, n_tot_iter))
    iter_counter += 1

loss_res_all = pd.concat(loss_res_all)
loss_res_all.to_csv(os.path.join(data_path, 'processed', '{}_scLEMBAS_model_losses.csv'.format(author)))

predictions_res_all = {k: v for d in predictions_res_all for k, v in d.items()}
io.write_pickled_object(predictions_res_all, 
                       os.path.join(data_path, 'processed', '{}_scLEMBAS_model_predictions.pickle'.format(author)))

predictions_res_noadv = {k: v for d in predictions_res_noadv for k, v in d.items()}
io.write_pickled_object(predictions_res_noadv, 
                       os.path.join(data_path, 'processed', '{}_scLEMBAS_noadv_predictions.pickle'.format(author)))

predictions_res_rand = {k: v for d in predictions_res_rand for k, v in d.items()}
io.write_pickled_object(predictions_res_rand, 
                       os.path.join(data_path, 'processed', '{}_scLEMBAS_rand_predictions.pickle'.format(author)))

Set up inputs for prediction


100%|█████████████████████████████████████████████| 4/4 [00:00<00:00,  4.69it/s]


Get the predictions


/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


Set up inputs for prediction


...

Set up inputs for prediction


100%|█████████████████████████████████████████████| 4/4 [00:00<00:00,  4.59it/s]


Get the predictions


/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


------------------ 44 of 45 ------------------


# Psuedo-bulk losses

Predictions were actually generated in the training script, so here we just calculate the loss. 

In [17]:
def pb_pearson(y_pred, y_actual):
    r = np.array([
        stats.pearsonr(y_pred.values[i], y_actual.values[i])[0]
        for i in range(y_actual.shape[0])
    ])
    return np.mean(r)

In [18]:
from sklearn.metrics import root_mean_squared_error
from scipy import stats

def pb_pipeline(fold, baseline_type):

    y_pred_baseline_pert, y_pred_baseline_condition = pb_y_pred(fold, author, baseline_type, tf_adata, pert_col)

    # psueo-bulked actual data and losses
    y_actual_pert = distances.psuedobulk_adata(adata = tf_adata, groupby_col = pert_col)
    y_actual_pert = y_actual_pert.loc[y_pred_baseline_pert.index, :]

    # for sklearn rmse, need to take transpose to get per-sample mean
    pd_pert = pb_pearson(y_actual_pert, y_pred_baseline_pert)
    rmse_pert = root_mean_squared_error(y_actual_pert.T, 
                                        y_pred_baseline_pert.T, 
                                        multioutput = 'uniform_average')

    y_actual_condition = distances.psuedobulk_adata(adata = tf_adata, groupby_col = 'condition')
    y_actual_condition = y_actual_condition.loc[y_pred_baseline_condition.index, :]
    pd_condition = pb_pearson(y_actual = y_actual_condition, y_pred=y_pred_baseline_condition)
    rmse_condition = root_mean_squared_error(y_actual_condition.T, 
                                             y_pred_baseline_condition.T, multioutput = 'uniform_average')
    
    loss_res = pd.DataFrame(
        data = {'loss': [rmse_pert, pd_pert, rmse_condition, pd_condition],
                'loss_type': ['RMSE_{}'.format(pert_col), 'Pearson_{}'.format(pert_col), 
                             'RMSE_condition', 'Pearson_condition'], 
                'remove_type': ['none']*4, 
                'fold': [fold]*4, 
                'model_type': ['psuedobulk_{}_baseline'.format(baseline_type)]*4
               }
    )

    return loss_res



In [19]:
folds = range(5)
baseline_types = ['RF', 'linear','mean']

iterables = itertools.product(baseline_types, folds)

loss_res_pb = []
for (baseline_type, fold) in iterables:
    loss_res = pb_pipeline(fold, baseline_type)
    loss_res_pb.append(loss_res)
    
loss_res_pb = pd.concat(loss_res_pb)
loss_res_pb.to_csv(os.path.join(data_path, 'processed', '{}_psuedobulk_baseline_losses.csv'.format(author)))